In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from ripser import ripser
from persim import plot_diagrams
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定
OUT_DIR = "H6_Minority_Detection"
os.makedirs(OUT_DIR, exist_ok=True)

trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

# 不均衡データ（95% vs 5%）の設定
N_MAJORITY = 380
N_MINORITY = 20
N_TOTAL = N_MAJORITY + N_MINORITY
results = []

print("=== H6追加検証：隠蔽されたマイノリティ(Hidden Minority)のTDA検知 ===")

def clr_transform(X, epsilon=1e-10):
    X_eps = X + epsilon
    log_X = np.log(X_eps)
    return log_X - np.mean(log_X, axis=1, keepdims=True)

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(10, total_trials // N_TOTAL)
    print(f"\n[Step {step}/6] 試行: {total_trials:,} (D={D_features})")

    np.random.seed(42 + step)

    # --- A. 不均衡データの生成 ---
    # マジョリティ(95%)
    X_maj = np.random.uniform(0.01, 0.2, size=(N_MAJORITY, D_features))
    # マイノリティ(5%): マジョリティとは異なる分布
    X_min = np.random.uniform(0.7, 0.9, size=(N_MINORITY, D_features))

    X_chimera_raw = np.vstack([X_maj, X_min])
    X_null_raw = np.random.uniform(0.01, 0.2, size=(N_TOTAL, D_features)) # 完全な同質集団(Null)

    # --- B. ハブ（過分散）の注入 ---
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)

    X_chimera_skewed = X_chimera_raw.copy()
    X_null_skewed = X_null_raw.copy()

    X_chimera_skewed[:, :hub_dims] *= skew_multiplier
    X_null_skewed[:, :hub_dims] *= skew_multiplier

    # 組成データ（simplex）へ
    X_chimera_simplex = X_chimera_skewed / np.sum(X_chimera_skewed, axis=1, keepdims=True)
    X_null_simplex = X_null_skewed / np.sum(X_null_skewed, axis=1, keepdims=True)

    # --- C. UMAPによる「マイノリティの隠蔽」 ---
    dist_cos_chimera = pairwise_distances(X_chimera_simplex + 1e-10, metric='cosine')
    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_emb = reducer.fit_transform(dist_cos_chimera)

    # --- D. TDAによる監査（Aitchison空間） ---
    dist_aitch_chimera = pairwise_distances(clr_transform(X_chimera_simplex), metric='euclidean')
    dist_aitch_null = pairwise_distances(clr_transform(X_null_simplex), metric='euclidean')

    ph_chimera = ripser(dist_aitch_chimera, distance_matrix=True, maxdim=1)['dgms']
    ph_null = ripser(dist_aitch_null, distance_matrix=True, maxdim=1)['dgms']

    h0_deaths_chimera = np.sort(ph_chimera[0][:, 1][ph_chimera[0][:, 1] < np.inf])
    gap_chimera = h0_deaths_chimera[-1] if len(h0_deaths_chimera) > 0 else 1e-5

    h0_deaths_null = np.sort(ph_null[0][:, 1][ph_null[0][:, 1] < np.inf])
    gap_null = h0_deaths_null[-1] if len(h0_deaths_null) > 0 else 1e-5

    topological_snr = gap_chimera / gap_null

    results.append({
        'Step': step, 'Dimensions': D_features,
        'Gap_Null': gap_null, 'Gap_Chimera': gap_chimera,
        'Topological_SNR': topological_snr
    })

    # --- E. グラフ出力 ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Detecting Hidden Minority (5% vs 95%) at D={D_features}', fontsize=16)

    # 1. UMAP（視覚的吸収）
    axes[0].scatter(umap_emb[:N_MAJORITY, 0], umap_emb[:N_MAJORITY, 1], label='Majority (95%)', alpha=0.5, color='blue')
    axes[0].scatter(umap_emb[N_MAJORITY:, 0], umap_emb[N_MAJORITY:, 1], label='Minority (5%)', alpha=0.9, color='red', marker='x')
    axes[0].set_title('UMAP Illusion: Minority Absorbed')
    axes[0].legend()

    # 2. Null PD
    plot_diagrams(ph_null, ax=axes[1])
    axes[1].set_title(f'Null Model\nMax Noise Gap: {gap_null:.2f}')
    axes[1].axhline(y=gap_null, color='green', linestyle='--')

    # 3. 観測PD（マイノリティの告発）
    plot_diagrams(ph_chimera, ax=axes[2])
    axes[2].set_title(f'TDA Audit\nChimera Gap: {gap_chimera:.2f} (SNR: {topological_snr:.1f})')
    axes[2].axhline(y=gap_chimera, color='red', linestyle='--')
    axes[2].axhline(y=gap_null, color='green', linestyle=':')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/minority_step_{step}.png")
    plt.close()

# 4. 集計とまとめ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/minority_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(trial_scales, df_results['Topological_SNR'], marker='D', color='darkorange', linewidth=2, label='Topological SNR (Minority vs Majority)')
ax.axhspan(0, 2.0, color='lightgreen', alpha=0.3, label='Safe Zone (Homogeneous)')
ax.axhline(y=2.0, color='green', linestyle='--', label='Rejection Threshold (SNR=2.0)')
ax.set_xscale('log')
ax.set_xlabel('Total Trials (log scale)')
ax.set_ylabel('Topological Effect Size (SNR)')
ax.set_title('Detecting 5% Hidden Minority via Topological SNR')
ax.grid(True, which="both", ls="--", alpha=0.5)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/minority_snr_plot.png")
plt.close()

shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.3 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=e576ae2b267351159e0242838179a0010fb7905ff3bab3d5f9b84ffa84ba69b9
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H6追加検証：隠蔽されたマイノリティ(Hidden Minority)のTDA検知 ===

[Step 1/6] 試行: 1,000 (D=10)

[Step 2/6] 試行: 10,000 (D=25)

[Step 3/6] 試行: 100,000 (D=250)

[Step 4/6] 試行: 1,000,000 (D=2500)

[Step 5/6] 試行: 5,000,000 (D=12500)

[Step 6/6] 試行: 10,000,000 (D=25000)

実験完了。全データを H6_Minority_Detection.zip に圧縮しました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>